# Koopman-LUSI-Net (KL-Net): Cross-Subject BCI Benchmark on Google Colab T4 GPU

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ChayseWright/portfolio/blob/main/koopman_lusi_benchmark/koopman_lusi_colab_benchmark.ipynb)
![License: MIT](https://img.shields.io/badge/License-MIT-yellow.svg)
![PyTorch 2.0+](https://img.shields.io/badge/PyTorch-2.0%2B-ee4c2c.svg)

**Author**: Chayse Wright (BYU Neuromechanics Research Group)  
**Target Hardware**: NVIDIA T4 GPU Runtime (Free-tier Google Colab)  

---

### Abstract & Overview
This notebook provides the authoritative, hardware-accelerated Leave-One-Subject-Out (LOSO) few-shot ($k \le 5$ calibration trials per class) cross-validation benchmark of **Koopman-LUSI-Net** against literature BCI baselines:
- **EEGNet-4,2** (Lawhern et al., 2018)
- **ShallowFBCSPNet** (Schirrmeister et al., 2017)
- **Riemannian MDM** (Barachant et al., 2012)
- **Systematic Ablations** (Koopman-Only Net, LUSI-Only Net, Base CNN)

On Google Colab's NVIDIA T4 GPU, the full 63-model training/adaptation lifecycle completes in **under 4 minutes** (vs. 2+ hours on workstation CPU).

## 1. Hardware Inspection
Verify that the Google Colab runtime is configured with an NVIDIA GPU (ideally T4). If not, navigate to `Runtime > Change runtime type > Hardware accelerator > T4 GPU`.

In [ ]:
# Inspect GPU hardware accelerator
!nvidia-smi

import torch
print(f"PyTorch Version: {torch.__version__}")
print(f"CUDA Available:  {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"Device Name:     {torch.cuda.get_device_name(0)}")
    print(f"VRAM:            {torch.cuda.get_device_properties(0).total_memory / 1e9:.2f} GB")
else:
    print("[NOTICE] Running on CPU. Recommend switching to GPU for faster execution.")

## 2. Environment Setup & Repository Installation

In [ ]:
# Clone benchmark repository and install dependencies
import os
if not os.path.exists('koopman_lusi_benchmark'):
    # Try dedicated repo first, fallback to portfolio repository
    res = os.system('git clone https://github.com/ChayseWright/koopman_lusi_benchmark.git')
    if res != 0 or not os.path.exists('koopman_lusi_benchmark'):
        print('[NOTICE] Dedicated repo not found. Cloning from portfolio repository...')
        !git clone --depth 1 https://github.com/ChayseWright/portfolio.git
        !cp -r portfolio/koopman_lusi_benchmark .

%cd koopman_lusi_benchmark

# Install pinned dependencies and package in editable mode
!pip install -q -r requirements.txt
!pip install -q -e .

print('\n[SUCCESS] Environment configured successfully.')


## 3. Rapid Integrity Smoke Test (`--demo` Mode)
Executes a fast smoke test across 2 subjects for 2 pre-training and 2 adaptation epochs. Validates tensor operations, Cayley parameterization, and statistical routines in <60 seconds.

In [ ]:
# Run fast demo mode verification
!python run_benchmark.py --demo --device auto

## 4. Full Production BCI Competition IV-2a Benchmark
Executes full Leave-One-Subject-Out (LOSO) cross-validation across all 9 subjects with $k=5$ calibration trials per class on the NVIDIA T4 GPU.

In [ ]:
# Full 9-subject LOSO benchmark on BCI Competition IV-2a
!python run_benchmark.py --dataset bci_iv_2a --device cuda --k_shots 5 --output_dir ./results

## 5. Automated PhysioNet Motor Imagery Benchmark (Optional)
To evaluate the 109-subject PhysioNet dataset (or a subset), execute with `--dataset physionet`:

In [ ]:
# Evaluate subset of PhysioNet MI subjects (64 EEG channels @ 160 Hz)
# !python run_benchmark.py --dataset physionet --subjects 1 2 3 --device cuda --k_shots 5 --output_dir ./results_physionet

## 6. Scientific Results & Statistical Significance Visualization
Renders the publication-ready Markdown table, distribution boxplots, and Koopman eigenvalue spectrum.

In [ ]:
from IPython.display import display, Markdown, Image
from pathlib import Path

# 1. Display Markdown Benchmark Report Table
md_report = Path("./results/tables/benchmark_report.md")
if md_report.exists():
    display(Markdown(md_report.read_text(encoding="utf-8")))
else:
    print("No benchmark report table found at ./results/tables/benchmark_report.md")

# 2. Display Accuracy Distribution Figures
fig_dist = Path("./results/figures/accuracy_distributions.png")
if fig_dist.exists():
    print("\n--- Accuracy Distributions ---")
    display(Image(filename=str(fig_dist)))

# 3. Display Koopman Eigenvalue Spectrum
fig_eigs = Path("./results/figures/koopman_eigenvalues.png")
if fig_eigs.exists():
    print("\n--- Cayley Koopman Operator Discrete Eigenvalue Spectrum ---")
    display(Image(filename=str(fig_eigs)))

## 7. Package & Download Artifacts
Bundles the full `./results` folder (raw metrics JSON/CSV, publication LaTeX and Markdown tables, and 300 DPI figures) into a zip archive for download.

In [ ]:
import shutil
from pathlib import Path

zip_name = "koopman_lusi_benchmark_results"
shutil.make_archive(zip_name, "zip", "./results")
print(f"Artifact packaged into: {zip_name}.zip")

try:
    from google.colab import files
    files.download(f"{zip_name}.zip")
    print("Initiated browser download of benchmark artifacts.")
except (ImportError, Exception):
    print(f"Download ready at: {Path.cwd() / (zip_name + '.zip')}")